### Configuration, inputs & outputs

In [ ]:
from pipeline_utils import make_case2_paths
from pathlib import Path
from collections import Counter
import pandas as pd
from PIL import Image
from tqdm import tqdm

# Configure paths
paths = make_case2_paths()

PROJECT_DIR = paths["project"]
RAW_DIR = paths["datasets"] / "images"
PROC_DIR = paths["proc"]

print(f"Project directory: {PROJECT_DIR}")
print(f"Raw images directory: {RAW_DIR}")
print(f"Processed data directory: {PROC_DIR}")

# Output files
CLEAN_DIR = PROC_DIR / "images_clean"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
images_clean_qc_table = PROC_DIR / "images_clean_qc.csv"

print(f"Clean images directory: {CLEAN_DIR}")

# Cropping parameter: PDX plots carry a fixed title/metadata band at the top
TOP_TRIM = 100

### Cropping function

In [2]:
def process_pdx_figure(image_path, output_path, top_trim):
    """Trim the top metadata band off a raw PDX figure and save it as PNG."""
    with Image.open(image_path) as img:
        width, height = img.size

        # Only the "top" coordinate changes -- this removes the title band
        crop_box = (0, top_trim, width, height)
        trimmed_img = img.crop(crop_box)

        final_output = output_path.with_suffix(".png")
        trimmed_img.save(final_output, "PNG")

    return final_output

### Batch execution

In [3]:
image_files = sorted(RAW_DIR.glob("*.webp"))
print(f"Found {len(image_files)} raw images to process.")

failed_files = []
for img_path in tqdm(image_files, desc="Trimming figures"):
    out_path = CLEAN_DIR / img_path.stem
    try:
        process_pdx_figure(img_path, out_path, TOP_TRIM)
    except Exception as e:
        failed_files.append((img_path.name, str(e)))

print(f"Successfully processed: {len(image_files) - len(failed_files)}")
print(f"Failed: {len(failed_files)}")
for name, err in failed_files:
    print(f"  {name}: {err}")

Found 1300 raw images to process.


Trimming figures: 100%|██████████| 1300/1300 [00:08<00:00, 154.46it/s]


Successfully processed: 1300
Failed: 0


### QC: output count and image-dimension consistency

In [4]:
output_files = sorted(CLEAN_DIR.glob("*.png"))
expected_count = len(image_files) - len(failed_files)

print(f"Clean images written: {len(output_files)}")
assert len(output_files) == expected_count, (
    f"Expected {expected_count} clean images, found {len(output_files)}"
)

output_sizes = Counter()
for f in output_files:
    with Image.open(f) as img:
        output_sizes[img.size] += 1

print(f"Distinct output image sizes: {len(output_sizes)}")
for size, count in sorted(output_sizes.items()):
    print(f"  {size}: {count} images")

Clean images written: 1300
Distinct output image sizes: 1
  (744, 620): 1300 images


### QC summary for image preprocessing

In [5]:
images_clean_qc = pd.Series({
    "raw_images": len(image_files),
    "processed_images": len(output_files),
    "failed_images": len(failed_files),
    "distinct_output_sizes": len(output_sizes),
})
images_clean_qc.to_csv(images_clean_qc_table, header=["value"])

print(f"Saved image-preprocessing QC summary to {images_clean_qc_table}")
print(images_clean_qc)

Saved image-preprocessing QC summary to /Users/guanqiaofeng/Documents/BHK/XevaDB/Manuscript_gitrepo/XevaDB_case_study/data/procdata/2_drugSensitivity/images_clean_qc.csv
raw_images               1300
processed_images         1300
failed_images               0
distinct_output_sizes       1
dtype: int64
